# 🛠️ Notebook 2: Blackjack — Implementation


## 🛠️ Setup

```bash
cd 07-object-oriented-design/blackjack
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🎯 Goal

Fill in the class skeletons from Notebook 1 and play a full round. Build it in small, independently-testable pieces:

1. `Card` — the smallest unit.
2. `Deck` — a collection of `Card`s.
3. `Hand` — the tricky Ace rule lives here.
4. `Player` / `Dealer` — decision-making, polymorphic.
5. `Game` — wires it together.


### 1️⃣ `Card` — rank, suit, and point value


In [ ]:
from dataclasses import dataclass, field
from enum import Enum
import random

class Suit(Enum):
    HEARTS = "♥"; DIAMONDS = "♦"; CLUBS = "♣"; SPADES = "♠"

RANKS = ["A","2","3","4","5","6","7","8","9","10","J","Q","K"]

@dataclass(frozen=True)   # frozen = immutable — a card never changes
class Card:
    rank: str
    suit: Suit
    def value(self) -> int:
        if self.rank == "A":            return 11   # start high; Hand downgrades if needed
        if self.rank in ("J","Q","K"):  return 10
        return int(self.rank)
    def __repr__(self):
        return f"{self.rank}{self.suit.value}"

# Quick sanity check
print(Card("A", Suit.SPADES), "=", Card("A", Suit.SPADES).value())
print(Card("K", Suit.HEARTS), "=", Card("K", Suit.HEARTS).value())
print(Card("7", Suit.CLUBS), "=", Card("7", Suit.CLUBS).value())


### 2️⃣ `Deck` — 52 cards, shuffle, draw


In [ ]:
@dataclass
class Deck:
    cards: list[Card] = field(default_factory=list)

    def __post_init__(self):
        if not self.cards:
            self.cards = [Card(r, s) for s in Suit for r in RANKS]

    def shuffle(self):       random.shuffle(self.cards)
    def draw(self) -> Card:  return self.cards.pop()
    def __len__(self):       return len(self.cards)

d = Deck()
print("Fresh deck size:", len(d))
d.shuffle()
print("Top 3 cards after shuffle:", [d.draw() for _ in range(3)])
print("Deck size after drawing 3:", len(d))


### 3️⃣ `Hand` — the Ace rule, unit-testable in isolation

Notice how `Hand` doesn't know about `Player`, `Deck`, or `Game`. That's **separation of concerns** — we can test the Ace rule without running a game.


In [ ]:
@dataclass
class Hand:
    cards: list[Card] = field(default_factory=list)

    def add(self, c: Card):
        self.cards.append(c)

    def value(self) -> int:
        total = sum(c.value() for c in self.cards)
        aces  = sum(1 for c in self.cards if c.rank == "A")
        # Downgrade Aces from 11 → 1 one at a time, only as needed
        while total > 21 and aces:
            total -= 10
            aces  -= 1
        return total

    def is_bust(self) -> bool: return self.value() > 21
    def is_blackjack(self) -> bool:
        return len(self.cards) == 2 and self.value() == 21
    def __repr__(self):
        return f"{self.cards} = {self.value()}"

# Mini "tests" — no framework needed, just asserts
def H(*ranks): return Hand([Card(r, Suit.SPADES) for r in ranks])

assert H("A", "K").value() == 21          # Ace counts as 11 → natural 21
assert H("A", "K").is_blackjack()
assert H("A", "5", "9").value() == 15     # Ace had to drop to 1 (11+5+9=25 → bust)
assert H("A", "A", "9").value() == 21     # One Ace stays 11, one drops to 1
assert H("K", "Q", "5").value() == 25     # No Aces → really busts
assert H("K", "Q", "5").is_bust()
print("All hand-value assertions passed ✅")


### 4️⃣ `Player` and `Dealer` — polymorphism

`Game` will call `wants_hit()` without caring whether it's a human-ish player or the dealer. That's the power of polymorphism: same interface, different behavior.


In [ ]:
class Player:
    """Naive player: mirrors the dealer's rule (hit while < 17)."""
    def __init__(self, name: str):
        self.name = name
        self.hand = Hand()

    def wants_hit(self) -> bool:
        return self.hand.value() < 17


class Dealer(Player):
    """Casino rule: dealer MUST hit until the hand value is ≥ 17."""
    def __init__(self):
        super().__init__("Dealer")

    def wants_hit(self) -> bool:
        return self.hand.value() < 17


### 5️⃣ `Game` — orchestrates one round

`Game` is deliberately boring: it just deals cards and asks each seat if they want another. All the *interesting* rules live inside the smaller classes.


In [ ]:
class Game:
    def __init__(self, players: list[Player]):
        self.deck = Deck()
        self.deck.shuffle()
        self.players = players
        self.dealer  = Dealer()

    def _deal_initial(self):
        # Standard casino deal order: one card at a time, around the table, twice
        for _ in range(2):
            for p in self.players + [self.dealer]:
                p.hand.add(self.deck.draw())

    def _play_turn(self, seat: Player):
        while seat.wants_hit() and not seat.hand.is_bust():
            seat.hand.add(self.deck.draw())
        print(f"  {seat.name}: {seat.hand}")

    def play(self):
        self._deal_initial()
        print("— Players —")
        for p in self.players:
            self._play_turn(p)
        print("— Dealer —")
        self._play_turn(self.dealer)
        print("— Settlement —")
        self._settle()

    def _settle(self):
        d = self.dealer.hand.value()
        for p in self.players:
            pv = p.hand.value()
            if   p.hand.is_bust():              result = "BUST — dealer wins"
            elif self.dealer.hand.is_bust():    result = "WIN (dealer busted)"
            elif pv >  d:                       result = "WIN"
            elif pv == d:                       result = "PUSH"
            else:                               result = "LOSE"
            print(f"  {p.name} ({pv}) vs Dealer ({d}): {result}")

random.seed(0)  # reproducible demo — change or remove for real randomness
Game([Player("Alice"), Player("Bob")]).play()


## 🧪 Run it a few times

Different seeds produce different rounds. This kind of *seeded randomness* is a simple, powerful trick for making randomized code testable.


In [ ]:
for seed in [1, 2, 3]:
    print(f"\n===== Seed {seed} =====")
    random.seed(seed)
    Game([Player("Alice")]).play()


## 🧭 Design lessons to take away

- **SRP (Single Responsibility):** the Ace rule only lives in `Hand`.
- **Polymorphism:** `Game` treats `Player` and `Dealer` identically — it just calls `wants_hit()`.
- **Encapsulation:** `Deck` hides *how* cards are stored; callers only see `shuffle()` / `draw()`.
- **Testability:** each class can be tested without spinning up a whole game.

## ➡️ Next

Open **`03_extensions.ipynb`** to add betting, swap inheritance for the Strategy Pattern, and handle splits — real-world extensions that separate a toy from a shippable design.
